# <span style="color:#B3B3FF; font-weight:bold;"> ╔═══════════════════════════════════════════╗ </span>
# <span style="color:#B3B3FF; font-weight:bold;"> ║ CLEXNET Tutorial on synthetic data 🤖 🧪</span>
# <span style="color:#B3B3FF; font-weight:bold;"> ╚═══════════════════════════════════════════╝ </span>
---

> ![figure1](https://anonymous.4open.science/r/ClexNet-8F0D/tut_img/fig1.png) 

## <span style="color:green; font-weight:700; font-family:monospace;"> ╭─────────────────────────────────────────────────────────────╮ </span>
## <span style="color:red; font-weight:700; font-family:monospace;">   $\triangleright$ Imports   📦                                  </span>
## <span style="color:green; font-weight:700; font-family:monospace;"> ╰─────────────────────────────────────────────────────────────╯ </span>


In [13]:
from __future__ import annotations
from typing import Tuple, Dict, Optional
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize, Bounds, bisect

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import numpy as np, math, copy


# <span style="color:green; font-weight:700; font-family:monospace;"> ╔═══════════════════════════════════════════╗ </span>
# <span style="color:#B3B3FF;; font-weight:700; font-family:monospace;">   $\triangleright$ Synthetic data generation 🧪                                    </span>
# <span style="color:green; font-weight:700; font-family:monospace;"> ╚═══════════════════════════════════════════╝ </span>

<span style="color:#B3B3FF; font-weight:bold;"> Generate a synthetic ‘organ–offer’ dataset with </span>

* 🧑‍⚕️ ▸ patient covariates  $\mathbf{X}$   (confounder)  
* 🫀 ▸ organ features  $\mathbf{O}$   (treatment)  
* 🎯 ▸ binary outcome  $Y$   (acceptance)  
* 💡 ▸ explanatory directions  $R$   (refusal&nbsp;reasons)

> ![figure2](https://anonymous.4open.science/r/ClexNet-8F0D/tut_img/fig2.png) 
     
<span style="color:#B3B3FF; font-weight:bold;"> The script produces four pandas DataFrames </span>
   * rct_df        – unbiased randomised trial
   * obs_train  – observational training set after bias–inducing construction 
   * obs_val    - observational validation set after bias–inducing construction 
   * obs_test   - observational test set after bias–inducing construction


<span style="color:#B3B3FF; font-weight:bold;"> Usage </span>

```python
rct_df, obs_train, obs_val, obs_test, meta = make_dataset(
    n_obs=100, n_rct=4_000, d_x=5, d_o=5,
    p_keep=0.1, rho=0.9, target_acceptance=0.50, seed=42)
obs_train["y"].mean()      # ≈ target_acceptance
```
---

In [3]:
# ───────────────────────────────────────────────────────────────
# constants & RNG helpers
# ───────────────────────────────────────────────────────────────
EPS = 1e-9
RNG: np.random.Generator = np.random.default_rng()

def _set_seeds(seed: Optional[int]) -> None:
    if seed is not None:
        global RNG
        RNG = np.random.default_rng(seed)
        torch.manual_seed(seed)

# ───────────────────────────────────────────────────────────────
# tiny building blocks
# ───────────────────────────────────────────────────────────────
class L2Norm(nn.Module):
    """Feature‑wise L2 normalisation layer."""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(x, p=2, dim=1)


def _mlp(in_dim: int,
         width: int,
         depth: int,
         *,
         l2: bool = False,
         out_dim: int = 1,
         act = nn.ReLU(),
         act_last: bool = False,
         logit_scale: float = 1.0    
         ) -> nn.Sequential:
    """
    Build an MLP:  [Linear → act → (optional L2)] × depth  →  Linear(out_dim)
    Optionally multiply the final logits by `logit_scale`
    before the (optional) final sigmoid.
    """
    layers: list[nn.Module] = []
    d = in_dim
    for _ in range(depth):
        layers += [nn.Linear(d, width), act]
        if l2:
            layers.append(L2Norm())
        d = width

    # final linear layer
    layers.append(nn.Linear(d, out_dim))

    # --- NEW: widen / narrow the logits -------------------------------
    if logit_scale != 1.0:
        layers.append(Scale(logit_scale))
    
    # optional output activation (e.g. sigmoid for probabilities)
    if act_last:
        layers.append(nn.Sigmoid())

    return nn.Sequential(*layers)

class BilinearOutcome(nn.Module):
    """
    f_Y = 4·(  x·A  +  o·B  +  ⟨x⊗o , C⟩  )
    • All coefficients are frozen ↦ fully reproducible ground truth.
    """
    def __init__(self, d_x: int, d_o: int, *, rng=None):
        super().__init__()
        if rng is None:
            rng = torch.Generator().manual_seed(0)

        # linear parts
        self.A = nn.Parameter(torch.randn(d_x,  generator=rng) * 0.2,
                              requires_grad=False)
        self.B = nn.Parameter(torch.randn(d_o,  generator=rng) * 0.2,
                              requires_grad=False)

        # bilinear part
        self.C = nn.Parameter(torch.randn(d_x, d_o, generator=rng) * 0.4,
                              requires_grad=False)

        self.d_x, self.d_o = d_x, d_o

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x, o = z[:, :self.d_x], z[:, self.d_x:]          # shapes (N,d_x), (N,d_o)

        lin_x  = torch.matmul(x, self.A)                 # (N,)
        lin_o  = torch.matmul(o, self.B)                 # (N,)

        # element‑wise product then weighted sum ⇒  (N,d_x,d_o) · (d_x,d_o)
        bilin  = (x.unsqueeze(2) * o.unsqueeze(1) * self.C).sum(dim=(1, 2))

        # ---------------------------------------------------------------
        return 5*(lin_x + lin_o + bilin ).unsqueeze(1)

# ───────────────────────────────────────────────────────────────
# main generator
# ───────────────────────────────────────────────────────────────
def make_dataset(*,
                 n_obs: int = 100_000,
                 n_rct: int | None = None,
                 obs_train_frac: float = 0.7,   # NEW
                 obs_val_frac:   float = 0.15,   # NEW
                 d_x: int = 3,
                 d_o: int = 2,
                 target_acceptance: float = 0.03,
                 target_variance: float | None = 0.00025,
                 reasons: bool = True,
                 ipw_reasons: bool = False,
                 # ‑‑‑ Bias inducers ------------------------------------------------
                 rho: float = 0.0,              # confounding  (0 = none)
                 p_keep: float = 0.1,           # proportion of selection (β)
                 collider_gamma: float = 0.0,   # outcome‑dependent selection (γ)
                 seed: int | None = None,
                 trials: int = 5
                 ) -> Tuple[pd.DataFrame, pd.DataFrame,
                                pd.DataFrame, pd.DataFrame, Dict]:

    """
    Returns (rct_df, obs_df, meta) with extra bias controls:

        • rho            — correlation between X and O
        • p_keep         — proporition kept g’s logits 
        • collider_gamma — multiplies keep_prob by p_raw^γ  (γ)
    """
    # ------------------------------------------------ initialisation
    _set_seeds(seed)

    raw_obs_N = int(np.ceil(n_obs / max(p_keep, EPS)))    # safety against p_keep=0
    if n_rct is None:
        n_rct = n_obs          # keep the RCT size aligned with final Obs
    
    if n_rct is None:
        n_rct = n_obs
    in_dim = d_x + d_o
    g   = _mlp(in_dim, 16, 1, l2=True)   # selection net
    f_Y = BilinearOutcome(d_x, d_o) 
    # ------------------------------------------------ helper: correlated O
    def _sample_O(x: np.ndarray) -> np.ndarray:
        if rho == 0.0:
            return RNG.standard_normal((len(x), d_o))
        A   = RNG.standard_normal((d_o, d_x)) / np.sqrt(d_x)    # optional: linear correlation 
        eps = RNG.standard_normal((len(x), d_o))
        return rho * (x @ A.T) + np.sqrt(max(EPS, 1 - rho**2)) * eps

    # A) observational cohort before selection  ──────────────                      
    X0 = RNG.standard_normal((raw_obs_N, d_x))
    O0 = _sample_O(X0)
    Z0 = np.hstack([X0, O0])
    
    with torch.no_grad():
        logits = g(torch.from_numpy(Z0).float()).squeeze(1).numpy()
    
    # rank logits → fractional ranks in [0,1]
    ranks = np.argsort(np.argsort(logits))
    frac  = ranks / (len(ranks) - 1)
    
    tail = p_keep / 2.0
    keep_prob = np.zeros_like(logits, dtype=float)
    keep_prob[(frac < tail) | (frac > 1.0 - tail)] = 1.0     # deterministic slice
    
    # optional collider modulation (still only uses X,O via p_raw)
    if collider_gamma > 0:
        with torch.no_grad():
            p_raw = torch.sigmoid(f_Y(torch.from_numpy(Z0).float())).numpy().squeeze(1)
        keep_prob *= np.power(p_raw, collider_gamma)
        keep_prob /= np.maximum(keep_prob.max(), EPS)        # rescale ≤1
    
    # final stochastic keep mask
    keep_mask = RNG.binomial(1, keep_prob).astype(bool)
    X_obs, O_obs, Z_obs = X0[keep_mask], O0[keep_mask], Z0[keep_mask]

    # B) independent RCT cohort
    X_rct = RNG.standard_normal((n_rct, d_x))
    O_rct = _sample_O(X_rct) * 0 + RNG.standard_normal((n_rct, d_o))  # break any X‑O link
    Z_rct = np.hstack([X_rct, O_rct])

    # C) outcome model + scaling
    with torch.no_grad():
        z_obs_raw = f_Y(torch.from_numpy(Z_obs).float()).numpy()           #.squeeze(1)
        z_rct_raw = f_Y(torch.from_numpy(Z_rct).float()).numpy()           #.squeeze(1)
    # ---------- 1) standardise logits wrt OBSERVATIONAL set ----------                   #rescaling & find v (Ahmed M. Alaa)
    #z_mean = z_obs_raw.mean()
    #z_std  = z_obs_raw.std(ddof=0) + EPS          # avoid /0
    #z_obs  = (z_obs_raw - z_mean) / z_std         # now std ≈ 1
    #z_rct  = (z_rct_raw - z_mean) / z_std         # *same* scaling!
    z_obs = z_obs_raw
    z_rct = z_rct_raw

    # ---------- 2) probability link  p = 1 / (1 + v·exp(z)) ----
    def p_of_v(v, z):
        """
        Logistic link   p = 1 / (1 + v · exp(z + 0.5))
        • z can be either a torch.Tensor or a numpy.ndarray / float
        • result has the same backend as the input
        """
        if isinstance(z, torch.Tensor):
            return 1.0 / (1.0 + v * torch.exp(z))
        else:                           # NumPy / python‑float
            return 1.0 / (1.0 + v * np.exp(z))
    
    # ---------- 3) solve for v so that mean(p_obs) = 3 % -------------
    def mean_diff(log10_v):
        v = 10.0 ** log10_v
        return p_of_v(v, z_obs).mean() - target_acceptance
    
    log10_v = bisect(mean_diff, -12, 12, xtol=1e-6)   # v ∈ [10⁻¹²,10¹²]
    v = 10.0 ** log10_v
    print(f"calibrated v = {v:.4g}")
    
    # ---------- 4) final probabilities & outcomes --------------------
    p_obs = p_of_v(v, z_obs)
    p_rct = p_of_v(v, z_rct)

    # ---------------- NO SCALING -------------------------
    # p_obs = z_obs_raw
    # p_rct = z_rct_raw

    p_obs = np.clip(p_obs, EPS, 1.0 - EPS)
    p_rct = np.clip(p_rct, EPS, 1.0 - EPS)


    # Bernoulli 
    y_obs = RNG.binomial(1, p_obs).astype(np.float32)
    y_rct = RNG.binomial(1, p_rct).astype(np.float32)

    # --- majority‑vote labels -------------------------------------------
    # y_obs = (RNG.binomial(trials, p_obs) >= (trials // 2 + 1)).astype(np.float32)
    # y_rct = (RNG.binomial(trials, p_rct) >= (trials // 2 + 1)).astype(np.float32)
    
    # Cut-off
    # y_obs = (p_obs >= 0.5).astype(np.float32)
    # y_rct = (p_rct >= 0.5).astype(np.float32)
    # inside make_dataset -----------------------
    
    z = p_obs    
    print("Max‑min probability range:", z.max() - z.min())   # usually ≈ 0.25
    print("Bayes‑optimal AUROC (p vs. y):", roc_auc_score(y_obs, z))

    ## # ------------------------------------------------------------------
    ## # Counter‑factual direction d
    ## # ------------------------------------------------------------------
    p_rct = p_rct.squeeze()               # (n_rct,)

    if reasons:
        # ─────────── step 1 ───────────  draw a fresh, large RCT pool
        N_big = max(10_000, 10 * len(Z_obs))          # ≥10 k or 10× obs
        X_big = RNG.standard_normal((N_big, d_x))
        O_big = RNG.standard_normal((N_big, d_o))
        Z_big = np.hstack([X_big, O_big])
    
        # outcome probabilities in that pool
        with torch.no_grad():
            z_big = f_Y(torch.from_numpy(Z_big).float()).squeeze(1).numpy()
        p_big = p_of_v(v, z_big).squeeze()
    
        # keep only the "easy positives"
        Z_big_pos = Z_big[p_big > 0.50]
        if Z_big_pos.size == 0:
            d_all = np.zeros_like(Z_obs)
        else:
            if ipw_reasons:
                # ─────────── step 2 ───────────  propensity of (X,O) in OBS
                #
                # p̂_obs(x,o)  ←   Kernel-density over observational covariates
                #
                from sklearn.neighbors import KernelDensity
                # Silverman bandwidth ≈ σ · (4/(D+2)/N)^(1/(D+4))
                sigma = Z_obs.std(axis=0, ddof=0).mean()
                bw = sigma * (4/(Z_obs.shape[1]+2)/Z_obs.shape[0])**(1/(Z_obs.shape[1]+4))
        
                kde = KernelDensity(bandwidth=bw, kernel="gaussian")
                kde.fit(Z_obs)                                   # density of OBS
                log_p_obs = kde.score_samples(Z_big_pos)         # log-density (P,)
                w =    log_p_obs                                 # ∝ 1 / p̂_obs
                w /= w.sum()                                     # normalise
                print(len(w))
    
                print(max(w))
            else:
                w = np.full(len(Z_big_pos), 1/len(Z_big_pos))
    
            # ─────────── step 3 ───────────  IPW-weighted pairing
            N_obs, D = Z_obs.shape
            d_all    = np.zeros((N_obs, D), dtype=float)
            neg_idx  = np.where(y_obs == 0)[0]
    
            for i in neg_idx:
                j = RNG.choice(len(Z_big_pos), p=w)          # (IPW) sample
                delta = Z_big_pos[j] - Z_obs[i]
                nrm   = np.linalg.norm(delta)
                if nrm > 1e-12:
                    d_all[i] = delta / nrm                   # unit-norm direction
        
            # Now d_all holds “contrastive” directions for negatives, unit‐norm
    
    # D) build dataframes
    def _df(X_, O_, y_, p_, d_=None):
        cols_X = [f"X{i}" for i in range(d_x)]
        cols_O = [f"O{j}" for j in range(d_o)]
        mat    = np.hstack([X_, O_])
    
        if d_ is not None:
            cols_dX = [f"dX{i}" for i in range(d_x)]
            cols_dO = [f"dO{j}" for j in range(d_o)]
            mat     = np.hstack([mat, d_])
            cols    = cols_X + cols_O + cols_dX + cols_dO
        else:
            cols    = cols_X + cols_O
    
        df = pd.DataFrame(mat, columns=cols)
        df["y"], df["p_true"] = y_, p_
        return df

    obs_df = _df(X_obs, O_obs, y_obs, p_obs, d_all)
    rct_df = _df(X_rct, O_rct, y_rct, p_rct)   # no counter‑factuals for RCT
    
    test_frac = 1.0 - obs_train_frac - obs_val_frac
    assert 0 < test_frac < 1, "fractions must sum to < 1"
    
    obs_tr, obs_tmp = train_test_split(
        obs_df, test_size=(obs_val_frac + test_frac),
        stratify=obs_df["y"], random_state=seed)
    
    rel_val = obs_val_frac / (obs_val_frac + test_frac)  # rescale inside tmp
    obs_val, obs_te = train_test_split(
        obs_tmp, test_size=test_frac / (obs_val_frac + test_frac),
        stratify=obs_tmp["y"], random_state=seed)

    # E) meta
    meta = dict(
        n_obs_initial=raw_obs_N,         # raw cohort size
        n_obs_final=len(obs_df),         # should equal n_obs
        n_rct=len(rct_df),
        acceptance_rate_obs=float(obs_df.y.mean()),
        acceptance_rate_rct=float(rct_df.y.mean()),
        rho=rho,
        p_keep = p_keep,
        collider_gamma=collider_gamma,
        seed=seed
    )
    return rct_df, obs_tr, obs_val, obs_te, meta

# ───────────────────────────────────────────────────────────────
# quick demo
# ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    rct, obs_train, obs_val, obs_test, meta = make_dataset(
        n_obs=100, n_rct=100, d_x=5, d_o=5,
        p_keep=0.1, rho= 0.9, target_acceptance=0.50, seed=42)
    print("Rows – RCT:", len(rct), "| Observational train:", len(obs_train))
    print("Mean acceptance (obs.):", meta["acceptance_rate_obs"] * 100, "%")

calibrated v = 0.002993
Max‑min probability range: 1.0
Bayes‑optimal AUROC (p vs. y): 0.9795918367346939
Rows – RCT: 100 | Observational train: 69
Mean acceptance (obs.): 49.000000953674316 %


## <span style="color:green; font-weight:700; font-family:monospace;"> ╭─────────────────────────────────────────────────────────────╮ </span>
## <span style="color:#B3B3FF; font-weight:700; font-family:monospace;">   $\triangleright$ Some (optional) sanity checks ✅                    </span>
## <span style="color:green; font-weight:700; font-family:monospace;"> ╰─────────────────────────────────────────────────────────────╯ </span>

> <span style="color:#B3B3FF; font-weight:bold;"> $\triangleright$ Check for linear correlations in observational data <span>
 
> <span style="color:#B3B3FF; font-weight:bold;"> $\triangleright$ Columns with **dX** and **dO** represent refusal reason components  <span>

In [4]:
(obs_train.corr()
 .style
 .format(lambda v: f"{v:.2f}".lstrip("0").replace("-0.", "-."))
)

,X0,X1,X2,X3,X4,O0,O1,O2,O3,O4,dX0,dX1,dX2,dX3,dX4,dO0,dO1,dO2,dO3,dO4,y,p_true
X0,1.00,-.39,-.50,-.18,.27,.11,-.82,.74,.19,-.68,-.69,.16,.31,.19,-.29,-.10,.51,-.47,-.18,.37,-.27,-.32
X1,-.39,1.00,.21,.25,-.08,-.16,-.06,-.44,-.50,.45,.34,-.32,-.01,-.19,.22,.07,-.09,.29,.12,-.07,-.19,-.17
X2,-.50,.21,1.00,.16,-.10,-.28,.44,-.68,-.29,.17,.23,-.03,-.62,-.00,.17,.29,-.45,.49,.15,-.24,.15,.19
X3,-.18,.25,.16,1.00,-.10,-.08,-.05,.15,-.58,-.14,.19,.06,-.02,-.38,-.04,-.07,.02,-.14,.28,-.01,-.01,-.03
X4,.27,-.08,-.10,-.10,1.00,.10,-.20,.21,.38,-.27,-.43,.11,.31,.30,-.60,-.03,.11,-.33,-.20,.29,-.01,-.05
O0,.11,-.16,-.28,-.08,.10,1.00,-.11,.11,.14,.09,-.09,.20,-.01,.11,-.14,-.29,-.06,-.02,-.04,-.01,.21,.12
O1,-.82,-.06,.44,-.05,-.20,-.11,1.00,-.61,.11,.46,.53,-.09,-.30,-.05,.13,.14,-.57,.39,.08,-.33,.33,.41
O2,.74,-.44,-.68,.15,.21,.11,-.61,1.00,.20,-.57,-.48,.23,.49,.01,-.36,-.28,.50,-.68,-.04,.29,-.10,-.12
O3,.19,-.50,-.29,-.58,.38,.14,.11,.20,1.00,-.09,-.24,.02,.14,.40,-.34,.10,.05,-.21,-.27,.20,.16,.15
O4,-.68,.45,.17,-.14,-.27,.09,.46,-.57,-.09,1.00,.59,-.17,-.22,-.17,.36,-.04,-.45,.39,.11,-.38,.08,.07


> <span style="color:#B3B3FF; font-weight:bold;">  $\triangleright$  Check for *no* linear correlations in RCT data  <span>

In [5]:
rct.corr()

,X0,X1,X2,X3,X4,O0,O1,O2,O3,O4,y,p_true
X0,1.000000,-0.002627,-0.075370,-0.130734,0.016745,0.077699,0.016399,0.074675,0.053667,0.130479,-0.097638,-0.057750
X1,-0.002627,1.000000,0.103261,0.086268,-0.132750,-0.058979,-0.016125,0.213725,0.032977,0.049920,0.037812,0.032065
X2,-0.075370,0.103261,1.000000,-0.140698,-0.127439,-0.069949,-0.053864,0.004338,0.140707,-0.048141,0.214051,0.197393
X3,-0.130734,0.086268,-0.140698,1.000000,0.052518,0.004359,0.069841,-0.072863,-0.144494,-0.066560,-0.000639,-0.014940
X4,0.016745,-0.132750,-0.127439,0.052518,1.000000,0.187186,-0.054974,0.140020,-0.111543,-0.042247,0.131401,0.151383
O0,0.077699,-0.058979,-0.069949,0.004359,0.187186,1.000000,0.114277,-0.090048,-0.293805,0.176096,0.099677,0.182674
O1,0.016399,-0.016125,-0.053864,0.069841,-0.054974,0.114277,1.000000,-0.024544,-0.060130,-0.029049,0.030615,0.021663
O2,0.074675,0.213725,0.004338,-0.072863,0.140020,-0.090048,-0.024544,1.000000,0.015757,0.037291,-0.019482,-0.031928
O3,0.053667,0.032977,0.140707,-0.144494,-0.111543,-0.293805,-0.060130,0.015757,1.000000,0.127226,-0.007338,-0.001836
O4,0.130479,0.049920,-0.048141,-0.066560,-0.042247,0.176096,-0.029049,0.037291,0.127226,1.000000,-0.088915,-0.072913


> <span style="color:#B3B3FF; font-weight:bold;">  $\triangleright$    Check for directional refusal reasons 

In [6]:
# show the first few rows of the counter‑factual directions
d_cols = [c for c in obs_train.columns if c.startswith(("dX", "dO"))]

display(
    obs_train[d_cols].head(10)          # change 10 → any number of rows you’d like
)

,dX0,dX1,dX2,dX3,dX4,dO0,dO1,dO2,dO3,dO4
69,-0.619254,-0.265424,0.275122,0.377416,-0.203524,0.176934,-0.469569,-0.008574,-0.185382,0.016636
37,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
48,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
79,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
92,-0.186618,0.238531,0.546560,-0.087245,-0.243003,-0.315538,0.204894,-0.593520,-0.104610,0.195273
33,0.377864,0.240945,-0.078957,-0.264401,0.679720,-0.155435,-0.091330,-0.089513,0.253789,-0.395070
26,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
80,-0.584090,0.426004,0.312541,-0.328790,0.048635,-0.194655,0.065914,-0.181425,0.237796,-0.370826
67,0.565548,-0.039973,0.021313,-0.282370,0.428267,0.019813,-0.437899,0.255481,-0.111590,-0.380903


## <span style="color:green; font-weight:700; font-family:monospace;"> ╭─────────────────────────────────────────────────────────────╮ </span>
## <span style="color:#B3B3FF; font-weight:700; font-family:monospace;">   $\triangleright$ Generate synthetic dataset 💾                    </span>
## <span style="color:green; font-weight:700; font-family:monospace;"> ╰─────────────────────────────────────────────────────────────╯ </span>

In [30]:
rct_df, obs_tr, obs_val, obs_te, meta = make_dataset(
        n_obs=1_000, n_rct= 10_000, d_x=5, d_o=5, p_keep = 0.1, rho = 0.9, ipw_reasons = False, target_acceptance= 0.5, seed=42)

print(f"Observational rows kept = {len(obs_tr)+len(obs_val)+len(obs_te):,} "
      f"(accept rate={meta['acceptance_rate_obs']*100:.2f} %)")
print(f"RCT rows                = {len(rct_df):,}  "
      f"(accept rate={meta['acceptance_rate_rct']*100:.2f} %)")

calibrated v = 0.0007789
Max‑min probability range: 1.0
Bayes‑optimal AUROC (p vs. y): 0.990051641859107
Observational rows kept = 1,000 (accept rate=49.70 %)
RCT rows                = 10,000  (accept rate=75.29 %)


<style>
.rainbow {
  font-family: monospace;         /* keeps the box aligned */
  font-weight: bold;
  background: linear-gradient(
      90deg,
      #ff0040 0%,
      #ff7f00 16%,
      #ffee00 33%,
      #00bf00 50%,
      #009dff 66%,
      #6f00ff 83%,
      #ff1493 100%);
  -webkit-background-clip: text;  /* Chrome/Safari */
  background-clip: text;          /* Firefox */
  color: transparent;             /* the trick: fill text with gradient */
}
</style>

# <span style="color:magenta; font-weight:bold;"> ╔═══════════════════════════════════════════╗ </span>
# <span style="color:#B3B3FF; font-weight:bold;"> $\triangleright$  ClexNet Implementation ➜ sklearn‑compatible classifier  🤖  </span>
# <span style="color:magenta; font-weight:bold;"> ╚═══════════════════════════════════════════╝ </span>

> ![figure3](https://anonymous.4open.science/r/ClexNet-8F0D/tut_img/fig3.png) 

In [15]:
class GradRev(Function):
    @staticmethod
    def forward(ctx, x, l): ctx.l = l; return x.view_as(x)
    @staticmethod
    def backward(ctx, g):   return -ctx.l * g, None

class L2Norm(nn.Module):
    def forward(self, x): return F.normalize(x, p=2, dim=1)

def make_stack(in_dim, hidden_dim, n_hidden, *, l2_norm=False):
    layers = []
    for _ in range(n_hidden):
        layers += [nn.Linear(in_dim, hidden_dim), nn.ReLU()]
        if l2_norm: layers.append(L2Norm())
        in_dim = hidden_dim
    return nn.Sequential(*layers), in_dim

class _ClexNetCore(nn.Module):
    def __init__(self, in_dim, k, enc_depth, enc_w,
                 y_depth, y_w, c_depth, c_w):
        super().__init__()
        self.enc, rep = make_stack(in_dim, enc_w, enc_depth, l2_norm=True)
        self.hY , hY  = make_stack(rep, y_w, y_depth)
        self.oY  = nn.Linear(hY, 1)
        self.hC , hC  = make_stack(rep, c_w, c_depth)
        self.oC  = nn.Linear(hC, k)

    def forward(self, x, lamb):
        h    = self.enc(x)
        logY = self.oY(self.hY(h)).squeeze(1)
        logC = self.oC(self.hC(GradRev.apply(h, lamb)))
        return logY, logC

class ClexNetClassifier(BaseEstimator, ClassifierMixin):
    """
    sklearn‑style wrapper around the ClexNet training routine
    --------------------------------------------------------
    Parameters shown here mirror the hyper‑params in your notebook;
    feel free to expose more if you want to tune them.
    """
    def __init__(self,
                 d_x, d_o,
                 epochs        =100,
                 patience      =10,
                 lr            =1e-3,
                 batch_size    =256,
                 k_cluster     =5,
                 λ_adv         =0.1,
                 ρ_expl        =0.05,
                 cf_target     =0.5,
                 M             =100,
                 expl_ep       = 37,
                 device        ="cpu",
                 F_scale       =1.0, # for testing only
                 enc_depth     =2, enc_width =32,
                 y_depth       =1, y_width   =32,
                 c_depth       =1, c_width   =32):
        self.d_x, self.d_o = d_x, d_o
        self.epochs, self.patience, self.lr   = epochs, patience, lr
        self.batch_size, self.device          = batch_size, device
        self.k_cluster, self.λ_adv, self.ρ_expl= k_cluster, λ_adv, ρ_expl
        self.M, self.cf_target, self.expl_ep  = M, cf_target, expl_ep
        self.enc_depth, self.enc_width = enc_depth, enc_width
        self.y_depth, self.y_width             = y_depth, y_width
        self.c_depth, self.c_width             = c_depth, c_width

        self.F_scale = F_scale

    # ---- helper: build dataloaders ---------------------------------
    def _dl(self, X, y, d):
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.float32)[:,None],
                           torch.tensor(d, dtype=torch.float32))
        return DataLoader(ds, batch_size=self.batch_size, shuffle=True)

    # ---- sklearn API: fit ------------------------------------------
    def fit(self, X, y,
            X_val=None, y_val=None, d_tr=None, d_val=None):

        if d_tr is None:               # grab counter‑factual columns
            d_cols = [c for c in self.X_cols_ if c.startswith(("dX","dO"))]
            d_tr  = self.obs_tr_[d_cols].values.astype(np.float32)
            d_val = self.obs_val_[d_cols].values.astype(np.float32)

        d_feat       = self.d_x + self.d_o
        dl_train     = self._dl(X,        y,        d_tr)
        dl_val       = self._dl(X_val,    y_val,    d_val)

        self.model_  = _ClexNetCore(
            d_feat, self.k_cluster,
            self.enc_depth, self.enc_width,
            self.y_depth , self.y_width ,
            self.c_depth , self.c_width ).to(self.device)

        opt   = torch.optim.Adam(self.model_.parameters(), lr=self.lr)
        bce   = nn.BCEWithLogitsLoss()
        ce    = nn.CrossEntropyLoss()

        # ---------- ➀ gradient‑history containers ---------------------
        self._grad_hist = dict(bce=[], adv=[], cfx=[])   # three lists

        # pick one encoder parameter to monitor
        enc_weight = next(self.model_.enc.parameters())
        gnorm = lambda: enc_weight.grad.detach().norm().item()

        # empirical box bounds (train set)
        feat_min = torch.tensor(X.min(0), dtype=torch.float32, device=self.device)
        feat_max = torch.tensor(X.max(0), dtype=torch.float32, device=self.device)
        feat_mean = torch.tensor(X.mean(0), dtype=torch.float32, device=self.device)
        
        if self.F_scale != 1.0:
            centre   = feat_mean                           #  ← new centre
            feat_min = centre + self.F_scale * (feat_min - centre)
            feat_max = centre + self.F_scale * (feat_max - centre)

        # k‑means on organ features
        O_train = X[:, self.d_x:].astype(np.float32)          # ← cast once
        kmeans  = KMeans(n_clusters=self.k_cluster,
                         n_init=10,      # silences the future‑warning
                         random_state=0).fit(O_train)
        
        def organ_cluster_ids(batch_X: torch.Tensor) -> torch.Tensor:
            """
            map a minibatch to organ‑cluster IDs (torch.long on the correct device)
            """
            ids = kmeans.predict(
                    batch_X[:, self.d_x:].cpu()
                            .numpy()
                            .astype(np.float32)               # ← same dtype
                  )
            return torch.tensor(ids, dtype=torch.long, device=self.device)

        # counter‑factual sampler
        def sample_cf(Xb: torch.Tensor, db: torch.Tensor) -> torch.Tensor:
            """
            returns (B,M,D) or an **empty tensor** if a row has alpha_max==0
            """
            eps   = 1e-12
            step  = torch.full_like(db, float('inf'))
            step  = torch.where(db > 0, (feat_max - Xb) / (db + eps), step)
            step  = torch.where(db < 0, (feat_min - Xb) / (db - eps), step)
            alpha_max = torch.min(step, 1).values.clamp(min=0.)               # (B,)
        
            # rows where alpha_max==0 → will be skipped by the caller; return dummy
            u      = torch.rand(Xb.size(0), self.M, device=self.device)
            alpha  = u * alpha_max[:, None]
            return Xb[:, None, :] + alpha[:, :, None] * db[:, None, :]

        # -------------------  TRAINING LOOP  ------------------------------
        best_val, best_state, no_imp = float("inf"), None, 0
        for ep in range(1, self.epochs + 1):
            self.model_.train()
        
            # running sums for gradient monitoring
            g_bce = g_adv = g_cfx = 0.0
            n_batches = 0
        
            for Xb, yb, db in dl_train:
                Xb, yb, db = [t.to(self.device) for t in (Xb, yb, db)]
                opt.zero_grad()
        
                # single forward pass with GRL active (λ_adv)
                logY, _      = self.model_(Xb[:, :d_feat], 0.0)         # no gradient flip
                _,    logC   = self.model_(Xb[:, :d_feat], -1) # gradients reversed
        
                loss_bce = bce(logY, yb.squeeze(1))                       #  ❶  prediction
                loss_adv = ce (logC, organ_cluster_ids(Xb))               #  ❷  adversarial
                loss_cfx = torch.tensor(0.0, device=self.device)
                
                # ── (c) counter‑factual exploration (negatives with non‑zero d)
                loss_cfx = torch.tensor(0., device=self.device)

                if ep > self.expl_ep and self.ρ_expl > 0:
                    mask = (yb.squeeze(1) == 0) & (db.norm(dim=1) > 0)
                    if mask.any():
                        with torch.no_grad():
                            cf = sample_cf(Xb[mask, :d_feat], db[mask, :d_feat])   # (B,M,D)
                
                        B, M, D  = cf.shape
                        log_cf,_ = self.model_(cf.view(B*M, D), 0.0)               # GRL off
                        log_cf   = log_cf.view(B, M)
                
                        # probabilities for each counter-factual
                        p_cf = torch.sigmoid(log_cf)                               # (B,M)
                
                        # highest probability in each minibatch row
                        p_max, _ = p_cf.max(dim=1)                                 # (B,)
                
                        τ = self.cf_target                                         # your threshold
                
                        # -------- probability-weighted aggregate -----------------
                        # normalised weights   w_ij ∝ p_ij
                        w = p_cf / (p_cf.sum(dim=1, keepdim=True) + 1e-12)         # (B,M)
                
                        # weighted score:   s_i = Σ_j w_ij · p_ij   (∈[0,1])
                        p_w = (w * p_cf).sum(dim=1)                                # (B,)
                
                        # -------- one-sided loss: only rows where p_max < τ -------
                        active = p_max < τ
                        if active.any():
                            # log-barrier hinge on the weighted score
                            loss_vals = -torch.log(p_w[active].clamp_min(1e-12))   # (K,)
                            loss_cfx  = self.ρ_expl * loss_vals.mean()
        
                # ----------------- total OrganITE‑style loss -------------
                loss = loss_bce - self.λ_adv* loss_adv + loss_cfx
                loss.backward()
        
                # -------- gradient bookkeeping (encoder weight) ----------
                g_bce += loss_bce.detach().item()
                g_adv += loss_adv.detach().item()
                g_cfx += loss_cfx.detach().item()
                n_batches += 1
        
                opt.step()
        
            # store mean gradient magnitudes for the epoch
            self._grad_hist["bce"].append(g_bce / n_batches)
            self._grad_hist["adv"].append(g_adv / n_batches)
            self._grad_hist["cfx"].append(g_cfx / n_batches)
        
            # ---------------- validation ----------------
            self.model_.eval()
            with torch.no_grad():
                p_val, y_val_all = [], []
                for Xb, yb, _ in dl_val:
                    logits, _ = self.model_(Xb[:, :d_feat].to(self.device), 0.0)
                    p_val.append(torch.sigmoid(logits).cpu())
                    y_val_all.append(yb.squeeze(1).cpu())
                p_val = torch.cat(p_val).numpy()
                y_val_all = torch.cat(y_val_all).numpy()
                p_val = np.nan_to_num(p_val, nan=0.5, posinf=1.0, neginf=0.0)
                val_bce = log_loss(y_val_all, p_val)
        
            #print(f"ep{ep:03d} | val BCE {val_bce:.4f} "
            #      f"AUROC {roc_auc_score(y_val_all, p_val):.3f}")
        
            if val_bce < best_val - 1e-4:
                best_val, best_state, no_imp = val_bce, copy.deepcopy(self.model_.state_dict()), 0
            else:
                no_imp += 1
                if no_imp >= self.patience:
                    print("⏹ early stop"); break

        # --------------------------------------------------------------
        # ➂  quick plot after training
        # --------------------------------------------------------------
        # import matplotlib.pyplot as plt
        # plt.figure(figsize=(6,4))
        # for k, color in zip(["bce","adv","cfx"], ["tab:blue",
        #                                           "tab:orange",
        #                                           "tab:green"]):
        #     plt.plot(self._grad_hist[k], label=k.upper(), lw=2, color=color)
        # plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("L")
        # plt.title("Losses"); plt.legend(); plt.show()

        self.model_.load_state_dict(best_state)
        self.model_.eval()
        return self

    # ---- sklearn API: predict_proba -------------------------------
    def predict_proba(self, X):
        X = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits,_ = self.model_(X[:, :(self.d_x+self.d_o)], 0.0)
            p = torch.sigmoid(logits).cpu().numpy().ravel()
        return np.column_stack([1-p, p])

# <span style="color:magenta; font-weight:bold;"> ╔═══════════════════════════════════════════╗ </span>
# <span style="color:#B3B3FF; font-weight:bold;"> $\triangleright$  Model zoo  🎭                </span>
# <span style="color:magenta; font-weight:bold;"> ╚═══════════════════════════════════════════╝ </span>

<span style="color:#B3B3FF; font-weight:bold;"> $\triangleright$  Compare ClexNet with: <span>

> <span style="color:#B3B3FF; font-weight:bold;">  $\triangleright$ Logistic Regression <span>
>
> <span style="color:#B3B3FF; font-weight:bold;">  $\triangleright$ Random Forest <span>
>
> <span style="color:#B3B3FF; font-weight:bold;">  $\triangleright$ PatientNet <span>
>
> <span style="color:#B3B3FF; font-weight:bold;">  $\triangleright$ OrganITE <span>
>
> <span style="color:#B3B3FF; font-weight:bold;">  $\triangleright$ ClexNet ($\rho = 0$)  <span>


In [16]:
_set_seeds(50)

# -------------------------------------------------------------------
# 2)  train / test split on *observational* data
# -------------------------------------------------------------------
X_cols  = [c for c in obs_tr.columns if c.startswith(("X", "O"))]
d_cols  = [c for c in obs_tr.columns if c.startswith(("dX", "dO"))]

X_tr,  y_tr  = obs_tr [X_cols].values, obs_tr ["y"].values
X_val, y_val = obs_val[X_cols].values, obs_val["y"].values
X_te,  y_te  = obs_te [X_cols].values, obs_te ["y"].values
X_rct, y_rct = rct_df [X_cols].values, rct_df["y"].values

d_tr = obs_tr [d_cols].values.astype(np.float32)
d_val= obs_val[d_cols].values.astype(np.float32)

# ------------------------------------------------------------------
# 3)  model zoo 
# ------------------------------------------------------------------
models = {
    "Logistic Reg.": make_pipeline(StandardScaler(),
                                   LogisticRegression(max_iter=100, solver="lbfgs")),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=None,
                                            n_jobs=-1, random_state=0),
    "PatientNet": ClexNetClassifier(
                   d_x=5, d_o=5,
                   epochs=100, patience=30, lr=1e-3, λ_adv=0, ρ_expl=0,
                   device="cuda" if torch.cuda.is_available() else "cpu"),
    "ClexNet - expl only": ClexNetClassifier(
                   d_x=5, d_o=5,
                   epochs=1000, patience=30, lr=1e-3, λ_adv=0, ρ_expl=0.15,
                   device="cuda" if torch.cuda.is_available() else "cpu"),
    "OrganITE": ClexNetClassifier(
                   d_x=5, d_o=5,
                   epochs=1000, patience=30, lr=1e-3, λ_adv=0.1, ρ_expl=0,
                   device="cuda" if torch.cuda.is_available() else "cpu"),
    "ClexNet": ClexNetClassifier(
                   d_x=5, d_o=5,
                   epochs=1000, patience=30, lr=1e-3, λ_adv=0.15, ρ_expl=0.15,
                   device="cuda" if torch.cuda.is_available() else "cpu")
}

# tell sklearn which extra arrays to feed into .fit()
fit_params = {
    "PatientNet": {"X_val": X_val, "y_val": y_val,
                "d_tr": d_tr,  "d_val": d_val},
    "OrganITE": {"X_val": X_val, "y_val": y_val,
                "d_tr": d_tr,  "d_val": d_val},
    "ClexNet - expl only": {"X_val": X_val, "y_val": y_val,
                "d_tr": d_tr,  "d_val": d_val},
    "ClexNet": {"X_val": X_val, "y_val": y_val,
                "d_tr": d_tr,  "d_val": d_val}
}

# ------------------------------------------------------------------
# 4)  train all models
# ------------------------------------------------------------------
for name, mdl in models.items():
    print("Training:", name)
    mdl.fit(X_tr, y_tr, **fit_params.get(name, {}))
    _set_seeds(50)


# -------------------------------------------------------------------
# 4)  metric helper with bootstrap
# -------------------------------------------------------------------
def bootstrap_metrics(y, p, n_boot=300, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    BCE_mean   = log_loss(y, p, labels=[0, 1])
    AUROC_mean = roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan
    PR_mean    = average_precision_score(y, p)
    Brier_mean = brier_score_loss(y, p)

    bce_bs, auc_bs, pr_bs, bri_bs = [], [], [], []
    for _ in range(n_boot):
        y_b, p_b = resample(y, p, replace=True)
        bce_bs.append(log_loss(y_b, p_b, eps=1e-15, labels=[0, 1]))
        auc_bs.append(roc_auc_score(y_b, p_b) if len(np.unique(y_b)) > 1 else np.nan)
        pr_bs .append(average_precision_score(y_b, p_b))
        bri_bs.append(brier_score_loss(y_b, p_b))

    return {
        "BCE_mean":   BCE_mean,            "BCE_std":   np.nanstd(bce_bs,  ddof=1),
        "AUROC_mean": AUROC_mean,          "AUROC_std": np.nanstd(auc_bs, ddof=1),
        "PR_mean":    PR_mean,             "PR_std":    np.nanstd(pr_bs,  ddof=1),
        "Brier_mean": Brier_mean,          "Brier_std": np.nanstd(bri_bs, ddof=1)
    }

def instance_metrics(y, p):
    perrow_bce   = -(y * np.log(np.clip(p, 1e-15, 1.0))
                     + (1 - y) * np.log(np.clip(1 - p, 1e-15, 1.0)))
    perrow_brier = (p - y) ** 2
    return {
        "BCE_mean":   perrow_bce.mean(),   "BCE_std": perrow_bce.std(ddof=1),
        "AUROC_mean": roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "AUROC_std":  np.nan,
        "PR_mean":    average_precision_score(y, p),
        "PR_std":     np.nan,
        "Brier_mean": perrow_brier.mean(), "Brier_std": perrow_brier.std(ddof=1)
    }

# -------------------------------------------------------------------
# 5)  evaluate everything  +  ground‑truth rows
# -------------------------------------------------------------------
records = []

for name, mdl in models.items():
    #p_val  = mdl.predict_proba(X_val)[:, 1]
    p_te   = mdl.predict_proba(X_te )[:, 1]
    p_rct  = mdl.predict_proba(X_rct)[:, 1]

    #records.append({"Model": name, "Dataset": "Obs‑Val",  **instance_metrics(y_val, p_val)})
    records.append({"Model": name, "Dataset": "Obs‑Test", **instance_metrics(y_te,  p_te)})
    records.append({"Model": name, "Dataset": "RCT",      **instance_metrics(y_rct, p_rct)})

# ground‑truth baselines ---------------------------------------------
for ds_name, y_ds, p_ds in [#("Obs‑Val",  y_val,  obs_val["p_true"].values),
                            ("Obs‑Test", y_te,   obs_te ["p_true"].values),
                            ("RCT",      y_rct,  rct_df["p_true"].values)]:
    records.append({"Model": "Ground Truth", "Dataset": ds_name,
                    **instance_metrics(y_ds, p_ds)})

# -------------------------------------------------------------------
# 6)  pretty output
# -------------------------------------------------------------------
cols = ["Model","Dataset",
        "BCE_mean","BCE_std","AUROC_mean","AUROC_std",
        "PR_mean","PR_std","Brier_mean","Brier_std"]

results = (pd.DataFrame(records)[cols]
           .sort_values(["Model","Dataset"])
           .reset_index(drop=True))

display(results.style.format({
    "BCE_mean": "{:.4f}",  "BCE_std": "{:.4f}",
    "AUROC_mean": "{:.3f}","AUROC_std": "{:.3f}",
    "PR_mean": "{:.3f}",   "PR_std": "{:.3f}",
    "Brier_mean": "{:.4f}","Brier_std": "{:.4f}"
}))

Training: Logistic Reg.
Training: Random Forest
Training: PatientNet
Training: ClexNet - expl only
⏹ early stop
Training: OrganITE
⏹ early stop
Training: ClexNet
⏹ early stop


,Model,Dataset,BCE_mean,BCE_std,AUROC_mean,AUROC_std,PR_mean,PR_std,Brier_mean,Brier_std
0,ClexNet,Obs‑Test,0.2749,0.3667,0.970,nan,0.958,nan,0.0762,0.1289
1,ClexNet,RCT,0.3767,0.6901,0.872,nan,0.948,nan,0.1165,0.2212
2,ClexNet - expl only,Obs‑Test,0.2364,0.5762,0.962,nan,0.961,nan,0.0667,0.1936
3,ClexNet - expl only,RCT,0.4276,0.7994,0.845,nan,0.938,nan,0.1326,0.2635
4,Ground Truth,Obs‑Test,0.1171,0.3096,0.992,nan,0.992,nan,0.0366,0.1152
5,Ground Truth,RCT,0.0838,0.3311,0.994,nan,0.998,nan,0.0254,0.1081
6,Logistic Reg.,Obs‑Test,0.7074,0.3882,0.590,nan,0.558,nan,0.2540,0.1715
7,Logistic Reg.,RCT,0.8403,0.6854,0.597,nan,0.816,nan,0.2979,0.2545
8,OrganITE,Obs‑Test,0.2433,0.2894,0.979,nan,0.977,nan,0.0681,0.1142
9,OrganITE,RCT,0.4422,0.6958,0.840,nan,0.932,nan,0.1402,0.2338
